# Train the generative student

This notebook fine-tunes a small language model to write each quotation as compact JSON. It scores the model on the 1,000 validation documents with the same metrics as the teacher, then exports it to ONNX for the browser demo.

**Run it on a free GPU.** In Colab choose *Runtime → Change runtime type → T4 GPU*; on Kaggle turn on *GPU T4*. Then *Run all*. It needs no API key and costs nothing. Training is the slow part, roughly an hour on a T4, and scoring 1,000 documents adds about 15 minutes.

**What you get:** `generative-results.json` (every number with its denominator) and `generative-onnx.zip` (the model the demo loads). Download both from the file panel and attach them in the project thread.

Nothing here reads the frozen test set.

In [ ]:
import sys

# The branch with this notebook. After the pull requests merge, change it to "main".
REF = "build/generative-notebook"
BASE = "HuggingFaceTB/SmolLM2-360M-Instruct"  # 362M parameters, Apache-2.0, runs in transformers.js
SEED = 0
EPOCHS = 2

!git clone -q --depth 1 --branch {REF} https://github.com/Chi944/quote-distill
%cd quote-distill
!pip install -q -e ".[generative]"
sys.path.insert(0, "src")  # this kernel started before the install, so point it at the package directly

## 1. Build the training data

The generator is deterministic, so this rebuilds exactly the files described in `docs/student-data.md`. The hashes below must match.

In [ ]:
import hashlib
import json
from pathlib import Path

!python -m quote_distill.student.data --out data/generated/student

def load(name):
    return [json.loads(line) for line in open(f"data/generated/student/{name}.jsonl", encoding="utf-8")]

train_rows, dev_rows = load("train"), load("dev")
hashes = {n: hashlib.sha256(Path(f"data/generated/student/{n}.jsonl").read_bytes()).hexdigest()[:12] for n in ("train", "dev")}
print(len(train_rows), len(dev_rows), hashes)
assert hashes == {"train": "7bde65f87aa2", "dev": "c15dc3f9e84e"}, "data differs from docs/student-data.md"

## 2. Token lengths

The prompt holds the whole document; the target is the compact JSON. Training examples longer than the limit are left out and counted.

In [ ]:
from transformers import AutoTokenizer

from quote_distill.student.generative import MAX_LENGTH, features

tok = AutoTokenizer.from_pretrained(BASE)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
_, dev_too_long, dev_lengths = features(dev_rows, tok)
print("dev tokens:", dev_lengths, "| longer than", MAX_LENGTH, ":", dev_too_long)

## 3. Train

In [ ]:
import torch

from quote_distill.student.generative import train

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (this will be very slow)")
_, train_too_long, _ = features(train_rows, tok)
print("training documents left out for length:", train_too_long)
model = train(train_rows, tok, BASE, "generative", epochs=EPOCHS, seed=SEED)

## 4. Score on the validation set

Greedy decoding, no repair: output that isn't one compact JSON object counts as unparsed and wrong everywhere, as it did for the teacher.

In [ ]:
import time

from quote_distill.student.generative import generate, score


def summary(report):
    keys = ("exact_documents", "parsed", "schema_valid", "critical_fields", "invented", "arithmetic_agrees")
    out = {k: str(report[k]) for k in keys}
    out["line_items_f1"] = report["line_items"]["f1"]
    out["fields"] = {k: str(v) for k, v in report["fields"].items()}
    out["line_fields"] = {k: str(v) for k, v in report["line_fields"].items()}
    return out

model.config.use_cache = True
t0 = time.perf_counter()
outputs = generate(model.eval(), tok, dev_rows, batch_size=16)
gpu_seconds = time.perf_counter() - t0
torch_report = score(dev_rows, outputs)
print(json.dumps(summary(torch_report), indent=1))
print(f"{gpu_seconds:.0f} s for {len(dev_rows)} documents on this GPU")
Path("generative-dev-outputs.jsonl").write_text("".join(json.dumps({"id": r["id"], "output": o}) + "\n"
                                                         for r, o in zip(dev_rows, outputs, strict=True)))

## 5. Export to ONNX and score the exported model

The browser runs a quantised file on CPU. Generating on CPU is slow, so the exported models are scored on the first 100 validation documents, next to the PyTorch model on the same 100. Latency is this machine's CPU, one document at a time; the demo measures the browser separately.

In [ ]:
from quote_distill.student.generative import export_onnx, onnx_model

sizes = export_onnx("generative", "generative-onnx")
print({k: f"{v / 1e6:.0f} MB" for k, v in sizes.items()})

subset = dev_rows[:100]
results = {"pytorch_same_100": summary(score(subset, outputs[:100]))}
for name in ("model_q4.onnx", "model_quantized.onnx"):
    ort = onnx_model("generative-onnx", name)
    times, outs = [], []
    for r in subset:
        t0 = time.perf_counter()
        outs += generate(ort, tok, [r], batch_size=1, device="cpu")
        times.append((time.perf_counter() - t0) * 1000)
    times.sort()
    results[name] = {"report": summary(score(subset, outs)),
                     "same_output_as_pytorch": sum(a == b for a, b in zip(outs, outputs[:100], strict=True)),
                     "cpu_ms_per_document": {"median": round(times[len(times) // 2]),
                                             "p95": round(times[int(0.95 * (len(times) - 1))]), "n": len(times)}}
    print(name, results[name]["report"]["exact_documents"], results[name]["cpu_ms_per_document"])

## 6. Save the results

In [ ]:
import shutil
import subprocess

import transformers

commit = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
out = {
    "base": BASE, "ref": REF, "commit": commit, "seed": SEED, "epochs": EPOCHS, "max_length": MAX_LENGTH,
    "data_sha256_12": hashes, "train_documents": len(train_rows), "dev_documents": len(dev_rows),
    "train_left_out_for_length": str(train_too_long), "dev_tokens": dev_lengths,
    "pytorch": summary(torch_report), "failing_documents": torch_report["failing_documents"],
    "gpu_seconds_for_dev": round(gpu_seconds), "onnx": results, "file_bytes": sizes,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "versions": {"torch": torch.__version__, "transformers": transformers.__version__},
}
Path("generative-results.json").write_text(json.dumps(out, indent=1))
shutil.make_archive("generative-onnx", "zip", "generative-onnx")
print("Download generative-results.json and generative-onnx.zip from the file panel (they are in the quote-distill folder).")